# N045 · 链表基础与哨兵节点

**目标：** 安全修改头部和中间节点而不遗失链条。

**先修：** N009, N017。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** next重接；dummy；前驱；插入删除；对象身份。

## 从问题到算法

链表修改先找到“待修改节点的前驱”。删除头节点没有自然前驱，哨兵将其变成普通删除。课程实现用size记录长度，get越界返回−1；负插入位置按0处理，超过长度的插入不执行。节点身份和next结构比打印出的值更重要。

## 最小实现

**本章接口：** `remove_elements(head, val)`、`MyLinkedList`

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def nodes_to_list(head):
    values, seen = [], set()
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values

def remove_elements(head,val):
    dummy=ListNode(0,head); previous=dummy
    while previous.next:
        if previous.next.val==val: previous.next=previous.next.next
        else: previous=previous.next
    return dummy.next

class MyLinkedList:
    def __init__(self): self.dummy=ListNode(); self.size=0
    def get(self,index):
        if not 0<=index<self.size: return -1
        node=self.dummy.next
        for _ in range(index): node=node.next
        return node.val
    def addAtHead(self,val): self.addAtIndex(0,val)
    def addAtTail(self,val): self.addAtIndex(self.size,val)
    def addAtIndex(self,index,val):
        index=max(index,0)
        if index>self.size: return
        before=self.dummy
        for _ in range(index): before=before.next
        before.next=ListNode(val,before.next); self.size+=1
    def deleteAtIndex(self,index):
        if not 0<=index<self.size: return
        before=self.dummy
        for _ in range(index): before=before.next
        before.next=before.next.next; self.size-=1

## 正确性、前提与复杂度

previous前面的节点都已完成过滤，previous.next是下一个未处理节点。删除时不推进previous，防止漏掉连续待删节点。

**代价：** 删除 O(n)；按索引操作 O(n)，头插 O(1)；哨兵与指针辅助空间 O(1)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[6,1,6,2,6]
show_table(['原值','删除值','剩余链'],[(a,6,nodes_to_list(remove_elements(list_to_nodes(a),6)))])
linked=MyLinkedList(); rows=[]
for index,value in [(0,1),(1,3),(1,2)]:
    linked.addAtIndex(index,value); rows.append((index,value,nodes_to_list(linked.dummy.next)))
show_table(['插入位置','插入值','链'],rows)

原值,删除值,剩余链
"[6, 1, 6, 2, 6]",6,"[1, 2]"


插入位置,插入值,链
0,1,[1]
1,3,"[1, 3]"
1,2,"[1, 2, 3]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert nodes_to_list(remove_elements(list_to_nodes([6,1,6,2,6]),6))==[1,2]
assert remove_elements(list_to_nodes([6,6]),6) is None
assert remove_elements(None,6) is None
x=MyLinkedList(); x.addAtHead(1); x.addAtTail(3); x.addAtIndex(1,2)
assert x.get(1)==2; x.deleteAtIndex(1); assert x.get(1)==3
assert nodes_to_list(x.dummy.next)==[1,3] and x.size==2
x.deleteAtIndex(9); assert x.size==2
print('N045: 所有本章断言通过')

N045: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 不使用dummy再实现一次。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明两种写法边界复杂度。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 203. Remove Linked List Elements（canonical）
- 707. Design Linked List（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。